In [1]:
!pip install pandas numpy scikit-learn tensorflow joblib -q
print("install completed")

install completed


In [2]:
import pandas as pd
import numpy as np
import random
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import joblib
import os


np.random.seed(42)
random.seed(42)
tf.random.set_seed(42)


def calculate_bmr(age, height_cm, weight_kg, metabolic_profile):
    # Harris-Benedict Formula
    # PROFILE_1 = Male, PROFILE_2 = Female
    if metabolic_profile == 'PROFILE_1':
        return 88.362 + (13.397 * weight_kg) + (4.799 * height_cm) - (5.677 * age)
    else:
        return 447.593 + (9.247 * weight_kg) + (3.098 * height_cm) - (4.330 * age)


def calculate_tdee(bmr, activity_level):
    multipliers = {
        'SEDENTARY': 1.2, 'LIGHT': 1.375, 'MODERATE': 1.55,
        'ACTIVE': 1.725, 'VERY_ACTIVE': 1.9
    }
    return bmr * multipliers[activity_level]


def calculate_target_calories(tdee, goal):
    if goal == 'LOSE':
        return tdee * 0.85
    elif goal == 'GAIN':
        return tdee * 1.15
    else:
        return tdee


def calculate_workout_params(age, weight_kg, goal, activity_level, metabolic_profile):
    if age < 25:
        base_intensity = 0.5
    elif age < 40:
        base_intensity = 0.6
    else:
        base_intensity = 0.4

    if goal == 'GAIN':
        base_intensity += 0.15
    elif goal == 'LOSE':
        base_intensity += 0.05

    activity_bonus = {
        'SEDENTARY': -0.1, 'LIGHT': 0.0, 'MODERATE': 0.05,
        'ACTIVE': 0.1, 'VERY_ACTIVE': 0.15
    }
    base_intensity += activity_bonus[activity_level]

    intensity = base_intensity * random.uniform(0.9, 1.1)
    intensity = np.clip(intensity, 0.0, 1.0)

    if age < 30:
        split_choice = random.choices([1, 2], weights=[0.4, 0.6])[0]
    elif age < 50:
        split_choice = random.choices([0, 1, 2], weights=[0.2, 0.6, 0.2])[0]
    else:
        split_choice = random.choices([0, 1], weights=[0.8, 0.2])[0]

    return intensity, split_choice


def calculate_macros(calories, goal, diet_pref, weight_kg, workout_intensity):

    if goal == 'LOSE':
        protein_per_kg = 2.2
    elif goal == 'GAIN':
        protein_per_kg = 2.0
    else:
        protein_per_kg = 1.6

    if diet_pref == 'HIGH_PROTEIN':
        protein_per_kg *= 1.15
    elif diet_pref == 'VEGETARIAN':
        protein_per_kg *= 0.9

    protein_per_kg *= (1.0 + 0.20 * workout_intensity)

    protein_g = weight_kg * protein_per_kg

    fat_g = (calories * 0.25) / 9

    remaining_cals = calories - (protein_g * 4) - (fat_g * 9)
    carbs_g = max(0, remaining_cals / 4)

    carbs_g *= (1.0 + 0.12 * workout_intensity)

    protein_g *= random.uniform(0.95, 1.05)
    carbs_g   *= random.uniform(0.95, 1.05)
    fat_g     *= random.uniform(0.95, 1.05)

    return protein_g, carbs_g, fat_g


print("complete")

complete


In [3]:
def generate_training_data(num_samples=10000):
    data = []

    goals = ['LOSE', 'MAINTAIN', 'GAIN']
    activity_levels = ['SEDENTARY', 'LIGHT', 'MODERATE', 'ACTIVE', 'VERY_ACTIVE']
    diet_prefs = ['BALANCED', 'HIGH_PROTEIN', 'VEGETARIAN', 'NO_PREFERENCE']
    metabolic_profiles = ['PROFILE_1', 'PROFILE_2']

    normal_samples = int(num_samples * 0.9)


    for _ in range(normal_samples):
        age = random.randint(18, 60)
        height_cm = random.uniform(150, 200)
        weight_kg = random.uniform(45, 120)
        goal = random.choice(goals)
        activity = random.choice(activity_levels)
        diet_pref = random.choice(diet_prefs)
        metabolic_profile = random.choice(metabolic_profiles)

        if random.random() < 0.05:
            weight_kg *= random.uniform(0.92, 1.08)
            height_cm *= random.uniform(0.98, 1.02)

        bmr = calculate_bmr(age, height_cm, weight_kg, metabolic_profile)
        tdee = calculate_tdee(bmr, activity)
        target_calories = calculate_target_calories(tdee, goal)

        metabolic_variation = random.uniform(0.88, 1.12)
        target_calories *= metabolic_variation

        workout_intensity, workout_type = calculate_workout_params(
            age, weight_kg, goal, activity, metabolic_profile
        )
        intensity_variation = random.uniform(0.92, 1.08)
        workout_intensity *= intensity_variation
        workout_intensity = np.clip(workout_intensity, 0.0, 1.0)

        protein_g, carbs_g, fat_g = calculate_macros(
            target_calories, goal, diet_pref, weight_kg, workout_intensity
        )

        if diet_pref == 'HIGH_PROTEIN' and random.random() < 0.4:
            protein_g *= random.uniform(0.85, 0.95)
            carbs_g *= random.uniform(1.05, 1.15)
        elif diet_pref == 'VEGETARIAN' and random.random() < 0.3:
            protein_g *= random.uniform(0.9, 1.0)

        data.append({
            'age': int(age),
            'heightCm': round(height_cm, 2),
            'weightKg': round(weight_kg, 2),
            'activityLevel': activity,
            'goal': goal,
            'dietPref': diet_pref,
            'metabolicProfile': metabolic_profile,
            'caloriesKcal': int(target_calories),
            'proteinG': int(protein_g),
            'carbsG': int(carbs_g),
            'fatG': int(fat_g),
            'workoutIntensity': round(workout_intensity, 4),
            'workoutType': int(workout_type)
        })


    extreme_samples = num_samples - normal_samples

    for _ in range(extreme_samples):
        scenario = random.choice(['underweight', 'obese', 'elderly', 'athlete', 'beginner'])

        if scenario == 'underweight':
            age = random.randint(18, 35)
            height_cm = random.uniform(160, 185)
            weight_kg = random.uniform(42, 55)
            goal = 'GAIN'
            activity = random.choice(['SEDENTARY', 'LIGHT'])
            diet_pref = 'HIGH_PROTEIN'
            metabolic_profile = random.choice(metabolic_profiles)

        elif scenario == 'obese':
            age = random.randint(30, 55)
            height_cm = random.uniform(155, 180)
            weight_kg = random.uniform(90, 118)
            goal = 'LOSE'
            activity = random.choice(['SEDENTARY', 'LIGHT'])
            diet_pref = random.choice(['BALANCED', 'HIGH_PROTEIN'])
            metabolic_profile = random.choice(metabolic_profiles)

        elif scenario == 'elderly':
            age = random.randint(55, 60)
            height_cm = random.uniform(155, 175)
            weight_kg = random.uniform(55, 85)
            goal = random.choice(['MAINTAIN', 'LOSE'])
            activity = random.choice(['SEDENTARY', 'LIGHT'])
            diet_pref = random.choice(diet_prefs)
            metabolic_profile = random.choice(metabolic_profiles)

        elif scenario == 'athlete':
            age = random.randint(20, 35)
            height_cm = random.uniform(165, 195)
            weight_kg = random.uniform(65, 95)
            goal = random.choice(['GAIN', 'MAINTAIN'])
            activity = random.choice(['ACTIVE', 'VERY_ACTIVE'])
            diet_pref = 'HIGH_PROTEIN'
            metabolic_profile = random.choice(metabolic_profiles)

        else:  # beginner
            age = random.randint(18, 40)
            height_cm = random.uniform(155, 185)
            weight_kg = random.uniform(50, 95)
            goal = random.choice(goals)
            activity = 'SEDENTARY'
            diet_pref = 'NO_PREFERENCE'
            metabolic_profile = random.choice(metabolic_profiles)

        bmr = calculate_bmr(age, height_cm, weight_kg, metabolic_profile)
        tdee = calculate_tdee(bmr, activity)
        target_calories = calculate_target_calories(tdee, goal)

        metabolic_variation = random.uniform(0.85, 1.15)
        target_calories *= metabolic_variation

        workout_intensity, workout_type = calculate_workout_params(
            age, weight_kg, goal, activity, metabolic_profile
        )

        if scenario == 'elderly':
            workout_intensity *= random.uniform(0.7, 0.85)
            workout_type = 0
        elif scenario == 'athlete':
            workout_intensity *= random.uniform(1.05, 1.15)

        workout_intensity = np.clip(workout_intensity, 0.0, 1.0)

        protein_g, carbs_g, fat_g = calculate_macros(
            target_calories, goal, diet_pref, weight_kg, workout_intensity
        )

        if scenario == 'athlete':
            protein_g *= random.uniform(1.1, 1.2)
            carbs_g *= random.uniform(0.95, 1.0)

        data.append({
            'age': int(age),
            'heightCm': round(height_cm, 2),
            'weightKg': round(weight_kg, 2),
            'activityLevel': activity,
            'goal': goal,
            'dietPref': diet_pref,
            'metabolicProfile': metabolic_profile,
            'caloriesKcal': int(target_calories),
            'proteinG': int(protein_g),
            'carbsG': int(carbs_g),
            'fatG': int(fat_g),
            'workoutIntensity': round(workout_intensity, 4),
            'workoutType': int(workout_type)
        })

    return pd.DataFrame(data)


df = generate_training_data(10000)


print("\nThe first five lines of data:")
print(df.head())

print("\nStatistical data:")
print(df[['caloriesKcal', 'proteinG', 'carbsG', 'fatG', 'workoutIntensity']].describe())

print("\nTarget distribution:")
print(df['goal'].value_counts())

print("\nActivity Level Distribution:")
print(df['activityLevel'].value_counts())


The first five lines of data:
   age  heightCm  weightKg activityLevel      goal       dietPref  \
0   58    155.57    100.62         LIGHT      LOSE   HIGH_PROTEIN   
1   32    172.46     65.86         LIGHT      LOSE  NO_PREFERENCE   
2   20    186.49     85.22     SEDENTARY  MAINTAIN     VEGETARIAN   
3   41    158.13     71.65      MODERATE      GAIN       BALANCED   
4   21    161.45     47.41        ACTIVE  MAINTAIN     VEGETARIAN   

  metabolicProfile  caloriesKcal  proteinG  carbsG  fatG  workoutIntensity  \
0        PROFILE_1          2370       243     196    62            0.3818   
1        PROFILE_2          1581       162     140    42            0.6711   
2        PROFILE_2          2173       121     280    57            0.3514   
3        PROFILE_1          2634       164     347    71            0.6360   
4        PROFILE_1          2631        70     467    72            0.6612   

   workoutType  
0            0  
1            0  
2            1  
3            1  


In [4]:
le_activity = LabelEncoder()
le_goal = LabelEncoder()
le_diet = LabelEncoder()
le_metabolic = LabelEncoder()

df['activityLevel_encoded'] = le_activity.fit_transform(df['activityLevel'])
df['goal_encoded'] = le_goal.fit_transform(df['goal'])
df['dietPref_encoded'] = le_diet.fit_transform(df['dietPref'])
df['metabolicProfile_encoded'] = le_metabolic.fit_transform(df['metabolicProfile'])

feature_columns = [
    'age', 'heightCm', 'weightKg',
    'activityLevel_encoded', 'goal_encoded',
    'dietPref_encoded', 'metabolicProfile_encoded'
]

X = df[feature_columns].values

y_workout = df[['workoutIntensity', 'workoutType']].values
y_meal = df[['caloriesKcal', 'proteinG', 'carbsG', 'fatG']].values

# Standardisation
print("Standardising data")
scaler_X = StandardScaler()
scaler_y_workout = StandardScaler()
scaler_y_meal = StandardScaler()

X_scaled = scaler_X.fit_transform(X)
y_workout_scaled = scaler_y_workout.fit_transform(y_workout)
y_meal_scaled = scaler_y_meal.fit_transform(y_meal)

X_train, X_test, y_workout_train, y_workout_test, y_meal_train, y_meal_test = train_test_split(
    X_scaled, y_workout_scaled, y_meal_scaled,
    test_size=0.2, random_state=42
)

print(f"Training set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")
print(f"Target number of workout: {y_workout_train.shape[1]}")
print(f"Target number of meal: {y_meal_train.shape[1]}")

Standardising data
Training set size: (8000, 7)
Test set size: (2000, 7)
Target number of workout: 2
Target number of meal: 4


In [5]:
# workout model

#input (7): age, height, weight, activity level, fitness goal, dietary preference, and metabolic profile.
#output (2): workout_intensity, workout_type

def create_workout_model(input_dim):
    inputs = keras.Input(shape=(input_dim,), name='user_input')

    # Feature extraction
    x = layers.Dense(128, activation='relu', name='workout_dense_1')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Dense(64, activation='relu', name='workout_dense_2')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)

    # Exercise-specific
    x = layers.Dense(32, activation='relu', name='workout_features')(x)

    # Output
    outputs = layers.Dense(2, activation='linear', name='workout_output')(x)

    model = keras.Model(inputs=inputs, outputs=outputs)

    model.compile(
        optimizer='adam',
        loss='mse',
        metrics=['mae']
    )

    return model


workout_model = create_workout_model(X_train.shape[1])
print("\n workout model architecture:")
workout_model.summary()

early_stop_workout = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True
)

print("\n training the work model...")
workout_history = workout_model.fit(
    X_train, y_workout_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[early_stop_workout],
    verbose=1
)

# Evaluate
workout_loss, workout_mae = workout_model.evaluate(X_test, y_workout_test)
print(f"\n Workout Model training completed")
print(f"Test set MAE: {workout_mae:.4f}")


 workout model architecture:


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ user_input (InputLayer)         │ (None, 7)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ workout_dense_1 (Dense)         │ (None, 128)            │         1,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ workout_dense_2 (Dense)         │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ workout_features (Dense)        │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ workout_output (Dense)          │ (None, 2)              │            66 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 12,194 (47.63 KB)

 Trainable params: 11,810 (46.13 KB)

 Non-trainable params: 384 (1.50 KB)


 training the work model...
Epoch 1/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 10s 15ms/step - loss: 0.9856 - mae: 0.7850 - val_loss: 0.6860 - val_mae: 0.6796
Epoch 2/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - loss: 0.6862 - mae: 0.6724 - val_loss: 0.5549 - val_mae: 0.6159
Epoch 3/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - loss: 0.6190 - mae: 0.6402 - val_loss: 0.4919 - val_mae: 0.5726
Epoch 4/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - loss: 0.5723 - mae: 0.6140 - val_loss: 0.4648 - val_mae: 0.5534
Epoch 5/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - loss: 0.5441 - mae: 0.5982 - val_loss: 0.4444 - val_mae: 0.5386
Epoch 6/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.5153 - mae: 0.5802 - val_loss: 0.4328 - val_mae: 0.5271
Epoch 7/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.5042 - mae: 0.5719 - val_loss: 0.4204 - val_mae: 0.5162
Epoch 8/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.4833 - mae: 0.5600 - val_loss: 0.4114 - val_mae: 0.5089
Epoch 9/100
2

In [6]:
# meal model (based on workout model output)
#input(7+2): age, height, weight, activity level, fitness goal, dietary preference, and metabolic profile + workout_intensity, workout_type
#output(4): calories, protein, carbs, fat

def create_meal_model(input_dim):

    user_input = keras.Input(shape=(input_dim,), name='user_input')
    workout_input = keras.Input(shape=(2,), name='workout_input')
    combined = layers.Concatenate(name='combined_features')([user_input, workout_input])


    x = layers.Dense(128, activation='relu', name='meal_dense_1')(combined)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Dense(64, activation='relu', name='meal_dense_2')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Dense(32, activation='relu', name='meal_features')(x)

    # output
    outputs = layers.Dense(4, activation='linear', name='nutrition_output')(x)

    model = keras.Model(
        inputs=[user_input, workout_input],
        outputs=outputs
    )

    model.compile(
        optimizer='adam',
        loss='mse',
        metrics=['mae']
    )

    return model

y_workout_pred_train = workout_model.predict(X_train, verbose=0)
y_workout_pred_test = workout_model.predict(X_test, verbose=0)


meal_model = create_meal_model(X_train.shape[1])
print("\n meal model architecture:")
meal_model.summary()

early_stop_meal = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True
)

#training meal model
meal_history = meal_model.fit(
    [X_train, y_workout_pred_train],
    y_meal_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[early_stop_meal],
    verbose=1
)

# Evaluate
meal_loss, meal_mae = meal_model.evaluate(
    [X_test, y_workout_pred_test],
    y_meal_test
)
print(f"\n Meal model training completed")
print(f" test set MAE: {meal_mae:.4f}")


 meal model architecture:


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ user_input          │ (None, 7)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ workout_input       │ (None, 2)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ combined_features   │ (None, 9)         │          0 │ user_input[0][0], │
│ (Concatenate)       │                   │            │ workout_input[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ meal_dense_1        │ (None, 128)       │      1,280 │ combined_feature… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128)       │        512 │ meal_dense_1[0][… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 128)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ meal_dense_2        │ (None, 64)        │      8,256 │ dropout_2[0][0]   │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64)        │        256 │ meal_dense_2[0][… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 64)        │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ meal_features       │ (None, 32)        │      2,080 │ dropout_3[0][0]   │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ nutrition_output    │ (None, 4)         │        132 │ meal_features[0]… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 12,516 (48.89 KB)

 Trainable params: 12,132 (47.39 KB)

 Non-trainable params: 384 (1.50 KB)

Epoch 1/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - loss: 0.6872 - mae: 0.6403 - val_loss: 0.4485 - val_mae: 0.5294
Epoch 2/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.3718 - mae: 0.4768 - val_loss: 0.2005 - val_mae: 0.3486
Epoch 3/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.2981 - mae: 0.4250 - val_loss: 0.1472 - val_mae: 0.2990
Epoch 4/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.2595 - mae: 0.3946 - val_loss: 0.1353 - val_mae: 0.2846
Epoch 5/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.2425 - mae: 0.3812 - val_loss: 0.1258 - val_mae: 0.2746
Epoch 6/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.2207 - mae: 0.3655 - val_loss: 0.1232 - val_mae: 0.2717
Epoch 7/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.2105 - mae: 0.3556 - val_loss: 0.1163 - val_mae: 0.2648
Epoch 8/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.1976 - mae: 0.3450 - val_loss: 0.1161 - val_mae: 0.2628
Epoch 9/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/

In [7]:
print(" Evaluation of Cascade Model：")

# The complete cascade prediction process
y_workout_pred = workout_model.predict(X_test, verbose=0)
y_meal_pred = meal_model.predict([X_test, y_workout_pred], verbose=0)

# anti-standardisation
y_workout_original = scaler_y_workout.inverse_transform(y_workout_test)
y_workout_pred_original = scaler_y_workout.inverse_transform(y_workout_pred)

y_meal_original = scaler_y_meal.inverse_transform(y_meal_test)
y_meal_pred_original = scaler_y_meal.inverse_transform(y_meal_pred)


print("\n Workout model predictions (top 10 samples):")
workout_comparison = pd.DataFrame({
    'Actual strength': y_workout_original[:10, 0].round(2),
    'Predicted strength': y_workout_pred_original[:10, 0].round(2),
    'Actual type': y_workout_original[:10, 1].round(0).astype(int),
    'Predicted type': y_workout_pred_original[:10, 1].round(0).astype(int)
})
print(workout_comparison.to_string(index=False))

print("\n Meal model predictions (top 10 samples):")
meal_comparison = pd.DataFrame({
    'Actual calories': y_meal_original[:10, 0].astype(int),
    'Predicted Calories': y_meal_pred_original[:10, 0].astype(int),
    'Heat error': (y_meal_original[:10, 0] - y_meal_pred_original[:10, 0]).astype(int),
    'Actual protein': y_meal_original[:10, 1].round(1),
    'Predicted proteins': y_meal_pred_original[:10, 1].round(1)
})
print(meal_comparison.to_string(index=False))

from sklearn.metrics import mean_absolute_error, accuracy_score

strength_mae = mean_absolute_error(y_workout_original[:, 0], y_workout_pred_original[:, 0])
print(f"Workout Intensity MAE:  ±{strength_mae:.4f}")

type_acc = accuracy_score(y_workout_original[:, 1].round(0), y_workout_pred_original[:, 1].round(0))
print(f"Workout Type Accuracy: {type_acc*100:.2f}%")

calories_mae = mean_absolute_error(y_meal_original[:, 0], y_meal_pred_original[:, 0])
print(f"Calories Prediction MAE: ±{calories_mae:.2f} kcal")

protein_mae = mean_absolute_error(y_meal_original[:, 1], y_meal_pred_original[:, 1])
print(f"Protein Prediction MAE:  ±{protein_mae:.2f} g")

print("Cascade model training and evaluation completed")

 Evaluation of Cascade Model：

 Workout model predictions (top 10 samples):
 Actual strength  Predicted strength  Actual type  Predicted type
            0.57                0.52            1               1
            0.60                0.53            2               2
            0.54                0.55            0               0
            0.93                0.84            2               2
            0.98                0.90            2               2
            0.71                0.72            2               1
            0.53                0.56            0               0
            0.68                0.63            1               2
            0.66                0.62            2               1
            0.77                0.65            2               2

 Meal model predictions (top 10 samples):
 Actual calories  Predicted Calories  Heat error  Actual protein  Predicted proteins
            2271                2208          62           138.0      

In [8]:
import os
import joblib

save_dir = '/content/yxw1268/model'
os.makedirs(save_dir, exist_ok=True)

workout_model.save(f'{save_dir}/workout_model.keras')

meal_model.save(f'{save_dir}/meal_model.keras')

joblib.dump(scaler_X, f'{save_dir}/scaler_X.pkl')

joblib.dump(scaler_y_workout, f'{save_dir}/scaler_y_workout.pkl')

joblib.dump(scaler_y_meal, f'{save_dir}/scaler_y_meal.pkl')

joblib.dump(le_activity, f'{save_dir}/le_activity.pkl')
joblib.dump(le_goal, f'{save_dir}/le_goal.pkl')
joblib.dump(le_diet, f'{save_dir}/le_diet.pkl')
joblib.dump(le_metabolic, f'{save_dir}/le_metabolic.pkl')

print("\nAll files saved in .keras format")


All files saved in .keras format


In [9]:
def predict_cascaded(age, height, weight, activity, goal, diet, metabolic):

    activity_enc = le_activity.transform([activity])[0]
    goal_enc = le_goal.transform([goal])[0]
    diet_enc = le_diet.transform([diet])[0]
    metabolic_enc = le_metabolic.transform([metabolic])[0]

    X_input = np.array([[
        age, height, weight,
        activity_enc, goal_enc, diet_enc, metabolic_enc
    ]])

    X_scaled = scaler_X.transform(X_input)

    y_workout_pred = workout_model.predict(X_scaled, verbose=0)

    y_meal_pred = meal_model.predict([X_scaled, y_workout_pred], verbose=0)

    y_workout_original = scaler_y_workout.inverse_transform(y_workout_pred)
    y_meal_original = scaler_y_meal.inverse_transform(y_meal_pred)

    workout_type_map = {0: 'FBW', 1: 'UPPER_LOWER', 2: 'PPL'}
    workout_type_idx = int(round(y_workout_original[0][1]))
    workout_type_idx = np.clip(workout_type_idx, 0, 2)

    return {
        'caloriesKcal': int(y_meal_original[0][0]),
        'proteinG': round(float(y_meal_original[0][1]), 1),
        'carbsG': round(float(y_meal_original[0][2]), 1),
        'fatG': round(float(y_meal_original[0][3]), 1),

        'workoutIntensity': round(float(y_workout_original[0][0]), 2),
        'workoutType': workout_type_map[workout_type_idx]
    }


test_cases = [
    {
        "name": "25-year-old male，aiming to muscle gain",
        "age": 25, "height": 175, "weight": 70,
        "activity": "MODERATE", "goal": "GAIN",
        "diet": "HIGH_PROTEIN", "metabolic": "PROFILE_1"
    },
    {
        "name": "30-year-old female, aiming to lose weight",
        "age": 30, "height": 165, "weight": 65,
        "activity": "LIGHT", "goal": "LOSE",
        "diet": "BALANCED", "metabolic": "PROFILE_2"
    },
    {
        "name": "40-year-old male, aiming to maintain weight",
        "age": 40, "height": 178, "weight": 85,
        "activity": "ACTIVE", "goal": "MAINTAIN",
        "diet": "BALANCED", "metabolic": "PROFILE_1"
    }
]

for i, case in enumerate(test_cases, 1):
    name = case.pop('name')
    result = predict_cascaded(**case)
    metabolic_desc = "Male Metabolism" if case['metabolic'] == 'PROFILE_1' else "Female Metabolism"

    print(f"\ncase {i}: {name} ({metabolic_desc})")
    print(f"  input: {case['age']}years old, {case['height']}cm, {case['weight']}kg, {case['goal']}")
    print(f"  meal: {result['caloriesKcal']}calories | protein{result['proteinG']}g | carbohydrates{result['carbsG']}g | fat{result['fatG']}g")
    print(f"  workout: intensity{result['workoutIntensity']} | type{result['workoutType']}")


case 1: 25-year-old male，aiming to muscle gain (Male Metabolism)
  input: 25years old, 175cm, 70kg, GAIN
  meal: 3064calories | protein185.3g | carbohydrates431.9g | fat84.5g
  workout: intensity0.76 | typePPL

case 2: 30-year-old female, aiming to lose weight (Female Metabolism)
  input: 30years old, 165cm, 65kg, LOSE
  meal: 1679calories | protein160.4g | carbohydrates162.3g | fat46.2g
  workout: intensity0.65 | typeUPPER_LOWER

case 3: 40-year-old male, aiming to maintain weight (Male Metabolism)
  input: 40years old, 178cm, 85kg, MAINTAIN
  meal: 3128calories | protein154.9g | carbohydrates457.2g | fat86.2g
  workout: intensity0.63 | typeUPPER_LOWER


In [10]:
from google.colab import files
import os

model_dir = '/content/yxw1268/model'

files_to_download = [
    'workout_model.keras',
    'meal_model.keras',
    'scaler_X.pkl',
    'scaler_y_workout.pkl',
    'scaler_y_meal.pkl',
    'le_activity.pkl',
    'le_goal.pkl',
    'le_diet.pkl',
    'le_metabolic.pkl'
]

for filename in files_to_download:
    filepath = f'{model_dir}/{filename}'
    print(f"{filename}")
    files.download(filepath)

print("\nAll files downloaded")

workout_model.keras


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

meal_model.keras


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

le_metabolic.pkl


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


All files downloaded
